In [ ]:
import pathlib
fixed = 'import os, json, numpy as np\nfrom pathlib import Path\n# Слить ADFTD-RS и ADFTD-PS в ADFTD как в LEAD/data_preprocessing/ADFTD/\ndef merge(rs_dir, ps_dir, out_dir):\n    import json, numpy as np, os\n    def load_meta(folder):\n        with open(os.path.join(folder,"meta.json")) as f:\n            return json.load(f)\n    meta_rs=load_meta(rs_dir); meta_ps=load_meta(ps_dir)\n    assert meta_rs["T"]==meta_ps["T"] and meta_rs["C"]==meta_ps["C"]\n    T,C=meta_rs["T"], meta_rs["C"]\n    N_rs,N_ps=meta_rs["N"], meta_ps["N"]\n    N_total=N_rs+N_ps\n    print(f"N_rs={N_rs} N_ps={N_ps} N_total={N_total}")\n    Path(out_dir).mkdir(parents=True, exist_ok=True)\n    X_out=np.memmap(os.path.join(out_dir,"X.dat"), dtype="float32", mode="w+", shape=(N_total,T,C))\n    y_out=np.memmap(os.path.join(out_dir,"y.dat"), dtype="float32", mode="w+", shape=(N_total,3))\n    def copy_block(src, meta, dst_x, dst_y, off):\n        N=meta["N"]; T=meta["T"]; C=meta["C"]\n        X_src=np.memmap(os.path.join(src,"X.dat"), dtype="float32", mode="r", shape=(N,T,C))\n        y_src=np.memmap(os.path.join(src,"y.dat"), dtype="float32", mode="r", shape=(N,3))\n        dst_x[off:off+N]=X_src; dst_y[off:off+N]=y_src\n        del X_src,y_src\n        return off+N\n    cur=0\n    cur=copy_block(rs_dir, meta_rs, X_out, y_out, cur)\n    cur=copy_block(ps_dir, meta_ps, X_out, y_out, cur)\n    meta_out=dict(meta_rs); meta_out["N"]=int(N_total)\n    with open(os.path.join(out_dir,"meta.json"),"w") as f:\n        json.dump(meta_out,f,indent=2)\n    print(f"Saved merged to {out_dir}")\nif __name__=="__main__":\n    import sys\n    merge(sys.argv[1], sys.argv[2], sys.argv[3])\n'
import os, pathlib as pl, tarfile, shutil
base='/kaggle/input/datasets/asoksci/eeg-dementia-coursach-full'
prev='/kaggle/input/eeg-dementia-lead3'
out='/kaggle/working/repo'
import os as _os
_os.makedirs(out, exist_ok=True)
for tar in pl.Path(base).rglob('*.tar'):
    try:
        tf=tarfile.open(tar); tf.extractall(out); tf.close()
    except: pass
for p in ['configs','reports','figures']:
    s=pl.Path(base)/p
    if s.exists(): shutil.copytree(s, pl.Path(out)/p, dirs_exist_ok=True)
_os.makedirs(out+'/src', exist_ok=True)
for f in pl.Path(base).rglob('src/*.py'):
    shutil.copy(f, pl.Path(out)/'src'/f.name)
pl.Path(out+'/src/09_lead_merge.py').write_text(fixed)
import os as os2; os2.chdir(out)
pl.Path('src/09_lead_merge.py').write_text(fixed)
print('patched')
!ls -R /kaggle/input/eeg-dementia-lead3/repo/Processed/L400 2>&1 | head -n 100
!mkdir -p Processed/L400
!cp -r /kaggle/input/eeg-dementia-lead3/repo/Processed/L400/ADFTD-RS Processed/L400/ 2>&1 | head
!cp -r /kaggle/input/eeg-dementia-lead3/repo/Processed/L400/ADFTD-PS Processed/L400/ 2>&1 | head
!ls -lh Processed/L400/*/meta.json 2>&1 | head
!ls -lh Processed/L400/*/X.dat 2>&1 | head
!python src/09_lead_merge.py Processed/L400/ADFTD-RS Processed/L400/ADFTD-PS Processed/L400/ADFTD 2>&1 | tail -20
!ls -lh Processed/L400/ADFTD/ 2>&1 | head
!cat Processed/L400/ADFTD/meta.json
print('merge fixed done')
